#### Download dos relatórios em PDF por safra

In [1]:
!pip install PyPDF2
!pip install natsort

import os
import re
import csv
import requests

import time
from datetime import datetime, timedelta

from concurrent.futures import ThreadPoolExecutor, as_completed

import gc

import pandas as pd
import numpy as np
import pandas as pd

from PyPDF2 import PdfReader

from natsort import natsorted

StatementMeta(, dc197a7d-5e54-4efc-9dca-5fa492e3f352, 3, Finished, Available, Finished)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.2 MB/s eta 0:00:00


In [ ]:
def validar_safra(safra):
    if not re.fullmatch(r'\d{6}', safra):
        raise ValueError(f"Safra '{safra}' deve estar no formato YYYYMM.")
    try:
        datetime.strptime(safra, '%Y%m')
    except ValueError:
        raise ValueError(f"Safra '{safra}' não representa uma data válida.")

def gerar_lista_safras(safra_inicio, safra_fim):
    validar_safra(safra_inicio)
    validar_safra(safra_fim)
    data_inicio = datetime.strptime(safra_inicio, '%Y%m')
    data_fim = datetime.strptime(safra_fim, '%Y%m')
    if data_inicio > data_fim:
        raise ValueError("A safra inicial deve ser anterior ou igual à safra final.")

    safra_atual = data_inicio
    lista = []
    while safra_atual <= data_fim:
        lista.append(safra_atual.strftime('%Y%m'))
        ano = safra_atual.year + (safra_atual.month // 12)
        mes = (safra_atual.month % 12) + 1
        safra_atual = datetime(ano, mes, 1)
    return lista

def ultima_safra():
    hoje = datetime.today()
    primeiro_dia_mes_atual = datetime(hoje.year, hoje.month, 1)
    ultimo_mes = primeiro_dia_mes_atual - timedelta(days=1)
    return ultimo_mes.strftime('%Y%m')

def baixar_pdf(codigo, safra, url_base, pasta_safra):
    nome_arquivo_pdf = os.path.join(pasta_safra, f"unidade_{codigo}.pdf")
    if os.path.exists(nome_arquivo_pdf):
        return (safra, codigo, "Ignorado", "Arquivo existente.")
    url = f"{url_base}{codigo}&format=pdf"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        with open(nome_arquivo_pdf, 'wb') as f:
            f.write(response.content)
        return (safra, codigo, "Sucesso", "Download OK")
    except requests.RequestException as e:
        return (safra, codigo, "Erro", str(e))

def download_pdfs(safra_inicio, safra_fim="auto", unidade_inicio=1, unidade_fim=5200, diretorio="/lakehouse/default/Files/TJSP_PRODUTIVIDADE", max_threads=5):
    if safra_fim == "auto":
        safra_fim = ultima_safra()

    inicio_tempo = time.time()
    lista_safras = gerar_lista_safras(safra_inicio, safra_fim)
    codigos = list(range(unidade_inicio, unidade_fim + 1))
    resultados = []

    for safra in lista_safras:
        t0 = time.time()
        pasta_safra = os.path.join(diretorio, safra)
        os.makedirs(pasta_safra, exist_ok=True)
        
        #Cache dos arquivos existentes para evitar chamadas lentas
        arquivos_existentes = set(os.listdir(pasta_safra))
        
        url_base = (
            f"https://www.tjsp.jus.br/APP/ProdutividadePrimeiraInstancia/Report/RelatorioCorreicao"
            f"?anoMesInicial={safra}01&anoMesFinal={safra}01&codigoUnidade="
        )

        tarefas = []
        with ThreadPoolExecutor(max_workers=max_threads) as executor:
            for codigo in codigos:
                nome_arquivo = f"unidade_{codigo}.pdf"
                if nome_arquivo not in arquivos_existentes:
                    tarefas.append(executor.submit(baixar_pdf, codigo, safra, url_base, pasta_safra))
                else:
                    resultados.append((safra, codigo, "Ignorado", "Arquivo existente."))

            for future in as_completed(tarefas):
                resultados.append(future.result())

        print(f"✅ Safra {safra} concluída em {time.time() - t0:.1f} segundos.")
        gc.collect()

    duracao_total = time.time() - inicio_tempo
    sucesso = sum(1 for r in resultados if r[2] == "Sucesso")
    ignorado = sum(1 for r in resultados if r[2] == "Ignorado")
    erro = sum(1 for r in resultados if r[2] == "Erro")

    print(f"\n🎯 Download concluído em {duracao_total:.1f} segundos.")
    print(f"📄 Arquivos baixados com sucesso: {sucesso}")
    print(f"📦 Arquivos ignorados (já existiam): {ignorado}")
    print(f"❌ Erros no download: {erro}")


StatementMeta(, 4d7a55a3-58a2-4e99-9e5b-07afc7cf00a2, 5, Finished, Available, Finished)

#### **DOWNLOAD DOS RELATÓRIOS**

In [ ]:
safra_inicio = "202201"
safra_fim = "auto"
unidade_inicio = 1
unidade_fim = 5200
diretorio = "/lakehouse/default/Files/TJSP_PRODUTIVIDADE"
max_threads = 20

download_pdfs(
    safra_inicio=safra_inicio,
    safra_fim=safra_fim,
    unidade_inicio=unidade_inicio,
    unidade_fim=unidade_fim,
    diretorio=diretorio,
    max_threads=max_threads
)

#### **PARSING**

In [6]:
import os
import re
import time
import logging
import numpy as np
import pandas as pd
from PyPDF2 import PdfReader
from concurrent.futures import ThreadPoolExecutor, as_completed
from natsort import natsorted
from tqdm import tqdm
import multiprocessing

# --- Configuração de logging ---
logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO  # Mude para DEBUG se quiser mais detalhes
)
logger = logging.getLogger(__name__)

# --- Constantes ---
MATERIAS = [
    "CÍVEL", "CRIMINAL", "EXECUÇÃO FISCAL", "INFÂNCIA",
    "JUIZADO CRIMINAL", "JUIZADO ESPECIAL",
    "JUIZADO FAZENDA PÚBLICA", "SETOR FAZENDA PÚBLICA"
]

PADRAO_MATERIA = re.compile(
    r"Foro:\s*(.+?)\s+Unidade:\s*(.+?)\s+Matéria:\s*(" + "|".join(map(re.escape, MATERIAS)) + r")",
    flags=re.IGNORECASE
)

PADRAO_FEITOS = re.compile(r"(?i)\bTotal\s+de\s+Feitos\s+em\s+Andamento\s*:?\s*(\d[\d\.]*)")

# --- Função: Extração de texto do PDF ---
def extrair_texto_pdf(file_path):
    try:
        reader = PdfReader(file_path)
        texto = " ".join(
            t.replace('\n', ' ').strip()
            for page in reader.pages
            for t in [page.extract_text()]
            if t
        )
        return re.sub(r"Considerações\s+para\s+análise\s+dos\s+dados.*$", "", texto, flags=re.IGNORECASE)
    except Exception as e:
        logger.error(f"Erro ao extrair texto de {file_path}: {e}")
        return ""

# --- Função: Extração de blocos estruturados ---
def extrair_blocos_completo(texto):
    matches = list(PADRAO_MATERIA.finditer(texto))
    blocos = []

    for i, m in enumerate(matches):
        start_idx = m.start()
        end_idx = matches[i + 1].start() if i + 1 < len(matches) else len(texto)
        bloco = texto[start_idx:end_idx]

        try:
            feitos = PADRAO_FEITOS.search(bloco)
            feitos_tot = int(feitos.group(1).replace(".", "")) if feitos else np.nan
        except Exception as e:
            logger.warning(f"Erro ao extrair feitos_tot: {e}")
            feitos_tot = np.nan

        blocos.append({
            "FORO": m.group(1).strip(),
            "UNIDADE": m.group(2).strip(),
            "MATERIA": m.group(3).strip(),
            "REGEX": bloco.strip(),
            "FEITOS_TOTAL": feitos_tot
        })

    return blocos

# --- Função: Processa um único PDF ---
def processar_pdf(nome_arquivo, caminho_pdf, safra):
    texto = extrair_texto_pdf(caminho_pdf)
    if not texto.strip():
        return []

    blocos = extrair_blocos_completo(texto)
    for bloco in blocos:
        bloco["SAFRA"] = safra
        bloco["ARQUIVO"] = nome_arquivo

    return blocos

# --- Função: Processa em paralelo todas as safras ---
def processar_pdfs_em_lote(pasta_base, max_threads=None):
    if not os.path.exists(pasta_base):
        raise FileNotFoundError(f"Pasta '{pasta_base}' não encontrada.")

    if max_threads is None:
        max_threads = max(1, multiprocessing.cpu_count() - 1)

    safras = [s for s in os.listdir(pasta_base) if os.path.isdir(os.path.join(pasta_base, s))]
    todos_dados = []
    inicio_total = time.time()

    for safra in safras:
        logger.info(f"📂 Processando safra: {safra}")
        inicio_safra = time.time()

        pasta_safra = os.path.join(pasta_base, safra)
        arquivos_pdf = natsorted([f for f in os.listdir(pasta_safra) if f.endswith(".pdf")])
        resultados = []

        with ThreadPoolExecutor(max_workers=max_threads) as executor:
            futuros = [
                executor.submit(processar_pdf, nome, os.path.join(pasta_safra, nome), safra)
                for nome in arquivos_pdf
            ]

            for futuro in tqdm(as_completed(futuros), total=len(futuros), desc=f"Processando {safra}", leave=False):
                resultado = futuro.result()
                if resultado:
                    resultados.extend(resultado)

        if resultados:
            df_safra = pd.DataFrame(resultados)
            df_safra["FEITOS_TOTAL"] = pd.to_numeric(df_safra["FEITOS_TOTAL"], errors="coerce").astype("Int64")
            todos_dados.append(df_safra)

        logger.info(f"✅ Safra {safra} processada: {len(arquivos_pdf)} arquivos.")
        logger.info(f"⏱️ Tempo safra: {time.time() - inicio_safra:.1f} segundos")

    if todos_dados:
        df_final = pd.concat(todos_dados, ignore_index=True)
        colunas = ["FORO", "UNIDADE", "MATERIA", "ARQUIVO", "REGEX", "SAFRA", "FEITOS_TOTAL"]
        df_final = df_final[colunas]
        logger.info(f"⏳ Tempo total: {time.time() - inicio_total:.1f} segundos")
        return df_final
    else:
        logger.warning("⚠️ Nenhum dado foi consolidado.")
        return pd.DataFrame()


StatementMeta(, 4d7a55a3-58a2-4e99-9e5b-07afc7cf00a2, 9, Finished, Available, Finished)

In [ ]:
df = processar_pdfs_em_lote("/lakehouse/default/Files/TJSP_PRODUTIVIDADE", max_threads=10)

# Normaliza nomes das colunas para snake_case
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

#### **APAGAR TABELA ANTIGA**

In [ ]:
spark.sql("DROP TABLE IF EXISTS tjsp_produtividade")
print("Tabela TJSP_Produtividade apagada com sucesso.")

#### **ARMAZENAMENTO NO LAKEHOUSE**

In [ ]:
if not df.empty:
    print(f"📊 Total de registros a serem gravados: {len(df)}")
    
    # Converte para Spark DataFrame
    df_spark = spark.createDataFrame(df)

    # Grava na tabela Delta no Fabric
    df_spark.write.mode("overwrite").saveAsTable("tjsp_produtividade")
    
    print("✅ Dados gravados na tabela 'tjsp_produtividade_teste' com sucesso.")
else:
    print("⚠️ Nenhum dado válido processado. Nada foi salvo.")
